# 06 · The shared subspace: what do the EMG and the hand have in common?

**Investigation ladder** (see [`EVALUATION.md`](../EVALUATION.md)):
- **Stage 4 · Link to movement: what does EMG encode (angle, velocity, change), at what lag?**

**Read first:** Semedo et al. 2019, *Cortical areas interact through a communication subspace* (Neuron); Gokcen et al. 2022, *Disentangling the flow of signals between populations of neurons* (DLAG, Nat Comput Sci)

**We expect:** a **small** set of EMG directions (1–3) carries almost all of the information about the pose, those directions are **not** the high-variance NMF synergies, and the four fingers share one direction while the thumb has its own. **Wrong if:** pose prediction keeps improving up to rank 16, or the shared directions are the same as the synergies.

**Stage 4b · Shared subspace** (between stage 4, *what does EMG encode?*, and stage 5, *dynamics*).
Stage 4 asked "how well can we estimate the pose?". This notebook asks a different question: **which latent
variables do the EMG and the pose have in common?**, e.g. a "finger flexion" signal visible in both. Instead of a
20-angle prediction, the answer is a few **shared latents**: directions in EMG space, the pose movements they
drive, and the delay between the two. They feed stage 5 (dynamics of the shared latents) and stage 6 (a
bottleneck chosen for pose, not for EMG variance).

**How this notebook works** (the same in every notebook)
- **`# TODO`** cells are yours. Each has a docstring with the input and output shapes. You write the core
  algorithms yourself in NumPy.
- Under every TODO there are three **💡 hints** that open one at a time: the plan in words, the tools with a tiny
  example, then a skeleton with blanks (`____`).
- After each TODO a **✅ check cell** compares your answer with the reference in [`ref06.py`](ref06.py), then a **📊 plot**
  shows what your code did. Other notebooks use the reference, so your blanks never break theirs.
- **✍️ questions** are for your own words, in the markdown cell under them.
- The **Result** cell at the end is the **gate**: it decides whether the next stage of the investigation is worth running.

## 🎯 Goal and milestones
*Draft milestones: agree them as a team on Friday, then edit here.*

**Proposal:** ➕ beyond the proposal, but it decides how the proposal's bottleneck (Aim 2) should be built.

**Goal:** find how many dimensions the EMG and the pose share, when the EMG leads, what each shared latent looks like, and whether these are the synergies from notebook 02.

**Why:** notebook 08's synergy bottleneck lost the pose information at k = 4. If the pose-relevant part of the EMG is a different, smaller subspace, the bottleneck should be built from it.

**Milestones**
1. A rank-vs-error curve on held-out stages, with a CI and a shifted-EMG null, and a chosen rank
2. The best delay of each shared latent
3. Principal angles between the shared subspace and the synergies, against a random-subspace null
4. Per-finger EMG directions and how similar they are
5. Pose-informed synergies compared with plain NMF and reduced-rank regression

**Gate (decides the next stage):** the shared subspace beats the shifted-EMG null on held-out stages with rank ≤ 3, and is distinguishable from the variance synergies (principal angles outside the random null). Pass → use the shared latents in stages 5 and 6.

**Done when:** a chosen rank with its CI, and a sentence on whether the shared subspace is the synergies.

In [ ]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    else:
        !git -C {ROOT} pull -q                 # pick up files pushed since the first clone
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

_d = ROOT / "notebooks" / "experiments" / "06_shared_subspace"
assert (_d / "ref06.py").exists(), f"{_d}/ref06.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref06", None)
_d = ROOT / "notebooks" / "experiments" / "02_how_many_synergies"
assert (_d / "ref02.py").exists(), f"{_d}/ref02.py is missing: pull the latest version of the repo"
sys.path.insert(0, str(_d))
sys.modules.pop("ref02", None)
_d = ROOT / "notebooks" / "experiments" / "common"
sys.path.insert(0, str(_d)); sys.modules.pop("protocol", None)
import protocol                 # shared evaluation protocol: sessions, folds, baselines, nulls, CIs
import ref06 as ref          # the reference answers, used only by the ✅ check cells

PROCESSED = ROOT / "data" / "processed_mini.npz"   # made by notebook 01's prep.py (data/ is never committed)

In [ ]:
# ── Load the shared data (rebuilds it with prep.py if missing or built by an older version) ──
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, str(ROOT / "notebooks" / "experiments" / "01_data_prep")); sys.modules.pop("prep", None)
import prep
if not prep.is_current(PROCESSED):
    subprocess.run([sys.executable, "notebooks/experiments/01_data_prep/prep.py"], check=True)
d = np.load(PROCESSED)
import ref02                               # notebook 02's NMF and PCA (reference versions; paste your own if you like)

emg        = d["emg"].astype(np.float64)   # (N, 16)  EMG envelope, 50 frames/s, non-negative
pose       = d["pose"].astype(np.float64)  # (N, 20)  joint angles (degrees), low-passed at 6 Hz
pose_valid = d["pose_valid"]               # (N,)     False where tracking failed (padded by 40 ms)
rec        = d["rec"]                      # (N,)     recording of each frame (0–29)
split      = d["split"]                    # (N,)     0 = train, 1 = test, -1 = gap
stage_of_rec, side_of_rec, filename_of_rec = d["stage"], d["side"], d["filename"]
train, test = split == 0, split == 1
tr, te = train & pose_valid, test & pose_valid   # frames with usable pose
SEED = 0
JOINT_NAMES = ["THUMB_CMC_FE", "THUMB_CMC_AA", "THUMB_MCP_FE", "THUMB_IP_FE",
               "INDEX_MCP_AA", "INDEX_MCP_FE", "INDEX_PIP_FE", "INDEX_DIP_FE",
               "MIDDLE_MCP_AA", "MIDDLE_MCP_FE", "MIDDLE_PIP_FE", "MIDDLE_DIP_FE",
               "RING_MCP_AA", "RING_MCP_FE", "RING_PIP_FE", "RING_DIP_FE",
               "PINKY_MCP_AA", "PINKY_MCP_FE", "PINKY_PIP_FE", "PINKY_DIP_FE"]
print(f"{len(emg)} frames | {tr.sum()} train / {te.sum()} test frames with valid pose | "
      f"{len(np.unique(protocol.block_of_rec(filename_of_rec)))} independent blocks")

### The variables you start with
| Name | Shape | What it is |
|---|---|---|
| `emg` | (N, 16) | EMG envelope, ≥ 0 |
| `pose` | (N, 20) | joint angles in degrees (low-passed at 6 Hz in `prep.py`) |
| `pose_valid` | (N,) | False where tracking failed, padded by 40 ms |
| `rec`, `split` | (N,) | recording of each frame; 0 = train, 1 = test, −1 = gap |
| `tr`, `te` | (N,) | train / test frames **with valid pose** |
| `stage_of_rec`, `side_of_rec`, `filename_of_rec` | (30,) | per recording; pass `filename=filename_of_rec` to every `protocol` fold function |

**Two ways of testing** are used below. *Leave-one-stage-out* (`protocol.leave_one_stage_out`, 14 folds) asks
"does this hold for a **new movement**?". The *time split* (`tr` / `te`) asks "does it hold **later in the same
movements**?" and is used where a single fit is enough. Letters: **D** = EMG features, **J** = 20 joints, **r** = rank.

## Before you run anything: your prediction

Write what you expect and why (2–3 sentences). Compare at the end. Surprises are the interesting part.

> *Prediction:*

---
## Task 1 · EMG features: log, and the past
Two facts from stage 4 shape the input:
- **EMG leads movement.** Muscles fire, then the hand moves, tens to hundreds of milliseconds later. So the pose at
  time *t* is compared with the EMG at *t − lag*. A lagged copy only takes EMG from **earlier in the same recording**:
  the first `lag` frames of a recording repeat its first frame, so nothing leaks in from the previous recording.
- **EMG noise grows with the signal.** Taking `log(envelope + 0.001)` evens that out, so loud bursts don't dominate.

At 50 frames/s, 10 frames = 200 ms.

In [ ]:
def log_emg(emg: np.ndarray, eps: float = 1e-3) -> np.ndarray:
    """(N, D) envelope → (N, D) log(envelope + eps)."""
    # TODO
    raise NotImplementedError


def lag_features(X: np.ndarray, rec: np.ndarray, lags: list[int]) -> np.ndarray:
    """Past copies of X, never crossing into the previous recording.

    Parameters
    ----------
    X    : (N, D) float, features
    rec  : (N,) int, recording of each frame
    lags : list of non-negative ints (frames); 0 = now

    Returns
    -------
    (N, D * len(lags)): column block i holds X from lags[i] frames earlier within the same recording.
    The first lags[i] frames of each recording repeat that recording's first frame.
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 1? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `log_emg` is one line.
2. `lag_features`: for each lag, make an empty array like `X`. Then for each recording, take its rows (`idx`), and
   write `X[idx]` shifted down by `lag` rows into those rows. Fill the first `lag` rows with the recording's first row.
3. Put the blocks side by side with `np.concatenate(..., axis=1)`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `idx = np.where(rec == r)[0]` → the row numbers of recording r, in time order.
- Shifting down by `lag`: `out[idx[lag:]] = x[:-lag]` (frame t gets frame t − lag). `lag == 0` needs no shift.

```python
x = np.array([[1.], [2.], [3.], [4.]]); lag = 2
out = np.empty_like(x); out[lag:] = x[:-lag]; out[:lag] = x[0]
out.ravel()     # array([1., 1., 1., 2.])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def log_emg(emg, eps=1e-3):
    return np.log(____ + eps)


def lag_features(X, rec, lags):
    blocks = []
    for lag in lags:
        out = np.empty_like(X)
        for r in np.unique(rec):
            idx = np.where(rec == r)[0]
            x = X[idx]
            if lag == 0:
                out[idx] = x
            else:
                out[idx[lag:]] = x[:-____]
                out[idx[:lag]] = x[____]
        blocks.append(out)
    return np.concatenate(blocks, axis=____)
```

</details>

In [ ]:
# ✅ Check Task 1
toy = np.arange(12.0).reshape(6, 2); toy_rec = np.array([0, 0, 0, 1, 1, 1])
assert np.array_equal(lag_features(toy, toy_rec, [0, 1]), ref.lag_features(toy, toy_rec, [0, 1])), \
    "lag_features differs: does it restart at each recording and repeat the first frame?"
assert np.allclose(log_emg(emg[:100]), ref.log_emg(emg[:100]))
X0 = log_emg(emg)                               # (N, 16) log-EMG now
X = lag_features(X0, rec, [10])                 # (N, 16) log-EMG 200 ms earlier: the main input below
print(f"✅ features ready: X {X.shape} (log-EMG from 200 ms earlier)")

# 📊 One channel, now and 200 ms earlier, next to a joint. What to look for: does the lagged EMG line up better with the joint?
m = rec == 0; t_ = np.arange(m.sum()) / 50
fig, ax = plt.subplots(2, 1, figsize=(14, 4), sharex=True)
ax[0].plot(t_, X0[m, 4], lw=0.7, label="log-EMG ch 4, now"); ax[0].plot(t_, X[m, 4], lw=0.7, label="200 ms earlier")
ax[0].legend(loc="upper right"); ax[1].plot(t_, np.where(pose_valid[m], pose[m, 9], np.nan), color="k", lw=0.8)
ax[1].set_ylabel(JOINT_NAMES[9]); ax[1].set_xlabel("time (s)"); ax[0].set_xlim(0, 20); plt.tight_layout(); plt.show()

✍️ **Q1.** Why must a lagged copy never take frames from the **previous** recording? What would a
model learn at the start of each recording if it did?

> *Your answer:*

---
## Task 2 · Reduced-rank regression: how many dimensions do EMG and pose share?
Ordinary (ridge) regression maps the 16 EMG features to the 20 angles with a 16 × 20 matrix `B`, which can use all
16 directions of the EMG. **Reduced-rank regression (RRR)** forces the map through only **r** dimensions:

$$\hat Y = X\,B_r,\qquad B_r = B\,V V^\top$$

1. Centre `X` and `Y` (subtract their means) and fit ridge: `B = solve(XcᵀXc + αI, XcᵀYc)`.
2. The fitted pose `Ŷ = Xc B` lives in at most 16 directions of pose space. Its SVD, `Ŷ = U S Vᵀ`, ranks them by how
   much predicted pose they carry; keep the top r right singular vectors `V` (J × r).
3. Project the map onto them: `B_r = B V Vᵀ`. Predictions: `(X − x̄) B_r + ȳ`.

`V` is the **pose side** of the shared latents (which movements EMG can predict). The **EMG side** is the top r left
singular vectors of `B_r` (`predictive_subspace`): the EMG directions the map actually reads. Neuroscience calls this a
**communication subspace** (Semedo et al. 2019): often small, and often not where most of the activity is.

**The test:** sweep r, score on held-out stages, compare with the median pose and with **circularly shifted EMG**
(each recording's EMG rolled in time, which keeps its own structure but breaks its alignment with the pose). Choose
the **smallest r whose error is statistically indistinguishable from the best** (paired CI over folds).

In [ ]:
def rrr_fit(X: np.ndarray, Y: np.ndarray, rank: int, alpha: float = 1.0
            ) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Reduced-rank ridge regression.

    Parameters
    ----------
    X : (T, D) float, EMG features;  Y : (T, J) float, pose;  rank : r;  alpha : ridge penalty

    Returns
    -------
    B_r    : (D, J) the rank-r map
    x_mean : (D,), y_mean : (J,) the means removed before fitting
    V      : (J, r) top-r right singular vectors of the fitted values Xc @ B (the pose side)
    """
    # TODO
    raise NotImplementedError


def rrr_predict(X: np.ndarray, B_r: np.ndarray, x_mean: np.ndarray, y_mean: np.ndarray) -> np.ndarray:
    """(T, D) → (T, J) predictions."""
    # TODO
    raise NotImplementedError


def predictive_subspace(B_r: np.ndarray, rank: int) -> np.ndarray:
    """(D, r) orthonormal EMG-side basis: the top-r left singular vectors of B_r."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 2? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Means, centred copies, ridge coefficients `B` (D × J).
2. SVD of the fitted values `Xc @ B`; keep the first `rank` rows of `Vt`, transposed, as `V`.
3. `B_r = B @ V @ V.T`. Predicting: centre with `x_mean`, multiply, add `y_mean`.
4. `predictive_subspace`: SVD of `B_r`, first `rank` columns of `U`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.linalg.solve(A, b)` solves A x = b (better than inverting A).
- `U, S, Vt = np.linalg.svd(M, full_matrices=False)`: rows of `Vt` are right singular vectors, columns of `U` left ones,
  both sorted by `S`, largest first.

```python
M = np.array([[3.0, 0.0], [0.0, 1.0]])
U, S, Vt = np.linalg.svd(M)     # S = [3, 1]; Vt[0] = [1, 0] is the strongest direction
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def rrr_fit(X, Y, rank, alpha=1.0):
    x_mean, y_mean = X.mean(axis=0), Y.mean(axis=0)
    Xc, Yc = X - x_mean, Y - y_mean
    B = np.linalg.solve(Xc.T @ Xc + alpha * np.eye(X.shape[1]), Xc.T @ ____)
    _, _, Vt = np.linalg.svd(Xc @ ____, full_matrices=False)
    V = Vt[:____].T
    return B @ V @ V.T, x_mean, y_mean, V


def rrr_predict(X, B_r, x_mean, y_mean):
    return (X - ____) @ B_r + ____


def predictive_subspace(B_r, rank):
    U, _, _ = np.linalg.svd(B_r, full_matrices=False)
    return U[:, :____]
```

</details>

In [ ]:
# ✅ Check Task 2a
B3, mx, my, V3 = rrr_fit(X[tr], pose[tr], 3)
r_B3, r_mx, r_my, r_V3 = ref.rrr_fit(X[tr], pose[tr], 3)
assert B3.shape == (16, 20) and V3.shape == (20, 3)
assert np.allclose(B3, r_B3, atol=1e-6), "B_r differs: centre, ridge, SVD of the FITTED values, then B V Vᵀ"
assert np.allclose(V3 @ V3.T, r_V3 @ r_V3.T, atol=1e-6), "V spans a different subspace"
assert np.allclose(rrr_predict(X[te], B3, mx, my), ref.rrr_predict(X[te], r_B3, r_mx, r_my))
Q3 = predictive_subspace(B3, 3)
assert np.allclose(Q3 @ Q3.T, ref.predictive_subspace(r_B3, 3) @ ref.predictive_subspace(r_B3, 3).T, atol=1e-6)
print(f"✅ RRR works. Rank 3, time split: {np.abs(pose[te] - rrr_predict(X[te], B3, mx, my)).mean():.2f}° "
      f"(median pose {np.abs(pose[te] - np.median(pose[tr], axis=0)).mean():.2f}°)")

### Task 2b · The rank sweep on held-out stages
Loop over the 14 leave-one-stage-out folds; in each, fit every rank on the train frames and record the mean absolute
error on the held-out stage, plus the median-pose baseline. Then the same for **shifted EMG** at ranks 1 and 3.
This takes about 20 s.

In [ ]:
ranks = [1, 2, 3, 4, 5, 6, 8, 12, 16]
fold_names: list[str] = []
errs = []                     # becomes (14, len(ranks)): held-out error of each rank in each fold
base = []                     # becomes (14,): median-pose error in each fold

for name, ftr, fte in protocol.leave_one_stage_out(stage_of_rec, rec, pose_valid, filename=filename_of_rec):
    # TODO: append name; append the median-pose error on fte (median of pose[ftr]);
    # TODO: for each rank, rrr_fit on (X[ftr], pose[ftr]), predict X[fte], append the mean absolute error
    ...
errs, base = np.array(errs), np.array(base)

# Null: the same with circularly shifted EMG (ranks 1 and 3)
X_shift = protocol.circular_shift_null(X, rec, np.random.default_rng(SEED))
errs_null = []                # becomes (14, 2)
# TODO: the same fold loop on X_shift for ranks [1, 3]
errs_null = np.array(errs_null)

chosen_rank: int | None = None   # TODO: smallest rank whose paired difference from the best rank has a CI with low end ≤ 0.1°

#### 💡 Stuck on Task 2b? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Inside the fold loop: one baseline number, then one error per rank (a list → one row of `errs`).
2. The null loop is the same with `X_shift` and two ranks.
3. Choosing the rank: find the best column (lowest mean). For each rank in order, bootstrap the per-fold difference
   `errs[:, i] - errs[:, best]` with `protocol.session_bootstrap_ci`; the first rank whose CI's low end is ≤ 0.1 wins.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `protocol.session_bootstrap_ci(values)` → `(mean, low, high)` over the folds.
- `np.abs(a - b).mean()` → mean absolute error over all frames and joints.

```python
errs_toy = np.array([[3.0, 2.0, 2.0], [3.2, 2.1, 2.0]])
errs_toy.mean(axis=0).argmin()     # 1 (ties go to the first)
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
for name, ftr, fte in protocol.leave_one_stage_out(stage_of_rec, rec, pose_valid, filename=filename_of_rec):
    fold_names.append(name)
    base.append(np.abs(pose[fte] - np.median(pose[____], axis=0)).mean())
    row = []
    for r in ranks:
        B, mx_, my_, _ = rrr_fit(X[ftr], pose[ftr], r)
        row.append(np.abs(pose[fte] - rrr_predict(X[____], B, mx_, my_)).mean())
    errs.append(row)
errs, base = np.array(errs), np.array(base)

# null: same loop with X_shift and ranks [1, 3] → errs_null

best = errs.mean(axis=0).argmin()
for i, r in enumerate(ranks):
    _, lo, hi = protocol.session_bootstrap_ci(errs[:, i] - errs[:, ____])
    if lo <= 0.1:
        chosen_rank = r
        break
```

</details>

In [ ]:
# ✅ Check Task 2b (about 5 s)
_, r_errs, r_base = ref.heldout_rrr_errors(X, pose, stage_of_rec, rec, filename_of_rec, pose_valid, [1, 3])
assert errs.shape == (14, len(ranks)), f"errs shape {errs.shape}, expected (14, {len(ranks)})"
assert np.allclose(base, r_base, atol=1e-6), "baseline differs: median of the TRAIN frames of each fold"
assert np.allclose(errs[:, [0, 2]], r_errs, atol=1e-6), "rank 1 / 3 errors differ from the reference"
assert errs_null.shape == (14, 2)
assert chosen_rank == ref.choose_rank(errs, ranks), f"chosen rank {chosen_rank}, reference {ref.choose_rank(errs, ranks)}"
i3 = ranks.index(3)
for label, diff in [("rank 3 − median pose", errs[:, i3] - base), ("rank 3 − shifted EMG", errs[:, i3] - errs_null[:, 1]),
                    (f"rank {chosen_rank} − full rank", errs[:, ranks.index(chosen_rank)] - errs[:, -1])]:
    m_, lo, hi = protocol.session_bootstrap_ci(diff)
    print(f"   {label:24s} {m_:+.2f}° [{lo:+.2f}, {hi:+.2f}]")
print(f"✅ chosen rank: {chosen_rank}")

# 📊 What to look for: where the curve flattens, and how far it sits below both reference lines.
fig, ax = plt.subplots(figsize=(7, 4))
ci = np.array([protocol.session_bootstrap_ci(errs[:, i])[1:] for i in range(len(ranks))])
ax.plot(ranks, errs.mean(axis=0), marker="o", label="EMG → pose, rank r")
ax.fill_between(ranks, ci[:, 0], ci[:, 1], alpha=0.2)
ax.axhline(base.mean(), ls="--", color="gray", label="median pose")
ax.plot([1, 3], errs_null.mean(axis=0), "x", color="C3", ms=10, label="shifted EMG (null)")
ax.axvline(chosen_rank, color="k", lw=0.8, ls=":")
ax.set_xlabel("rank r"); ax.set_ylabel("held-out-stage error (°)"); ax.legend(); plt.tight_layout(); plt.show()

✍️ **Q2.** How many dimensions do the EMG and the pose share, by this test? Compare the gain
from rank 0 (the median pose) to rank 1 with the gain from rank 1 to 16. What does that say about the "finger
activation" idea?

> *Your answer:*

---
## Task 3 · Does each shared latent have its own delay? (a light version of DLAG)
DLAG (Gokcen et al. 2022) finds latents shared by two populations, each with **its own delay**. A simple version:
fit RRR once (rank 3, time split), then for each shared latent *i*
- its **EMG side** is `z_i = (X0 − x̄) Q[:, i]`, using the **un-lagged** log-EMG `X0`,
- its **pose side** is `y_i = (pose − ȳ) V[:, i]`,

and slide `z_i` back in time by 0 … 300 ms. The lag with the highest correlation is that latent's delay.
(The fit used EMG at 200 ms, so that lag has a head start; a clear peak elsewhere is still informative.)

In [ ]:
def latent_lag_profile(X0: np.ndarray, Y: np.ndarray, rec: np.ndarray, valid: np.ndarray, Q: np.ndarray,
                       V: np.ndarray, x_mean: np.ndarray, y_mean: np.ndarray, lags: list[int]) -> np.ndarray:
    """Correlation between each latent's EMG side (taken lag frames earlier) and its pose side.

    Parameters
    ----------
    X0 : (N, D) un-lagged features;  Y : (N, J) pose;  rec : (N,);  valid : (N,) bool frames to score
    Q  : (D, r) EMG-side basis;  V : (J, r) pose-side basis;  x_mean (D,), y_mean (J,) from rrr_fit
    lags : list of frame lags

    Returns
    -------
    (len(lags), r) correlations over valid frames. Flip each column's sign so its largest-magnitude value is positive.
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 3? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Compute the EMG sides `z` (N × r) and pose sides `y` (N × r) for all frames at once.
2. For each lag, shift `z` with your `lag_features(z, rec, [lag])` (recordings stay separate) and correlate column
   *i* with `y[:, i]` on the valid frames.
3. Fix the signs: a latent and its negative are the same latent.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.corrcoef(a, b)[0, 1]` → the correlation of two 1-D arrays.
- For the sign: `cols = np.arange(r)`; `rows = np.abs(out).argmax(axis=0)`; `signs = np.sign(out[rows, cols])`.

```python
out = np.array([[0.1, -0.5], [0.3, -0.2]])
np.sign(out[np.abs(out).argmax(axis=0), np.arange(2)])    # array([ 1., -1.]) → flip column 1
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def latent_lag_profile(X0, Y, rec, valid, Q, V, x_mean, y_mean, lags):
    z = (X0 - x_mean) @ ____
    y = (Y - y_mean) @ ____
    out = np.zeros((len(lags), Q.shape[1]))
    for a, lag in enumerate(lags):
        zl = lag_features(z, rec, [____])
        for i in range(Q.shape[1]):
            out[a, i] = np.corrcoef(zl[valid, i], y[valid, ____])[0, 1]
    signs = np.sign(out[np.abs(out).argmax(axis=0), np.arange(Q.shape[1])])
    return out * signs
```

</details>

In [ ]:
# ✅ Check Task 3
lags = list(range(0, 16))                  # 0 … 300 ms
prof = latent_lag_profile(X0, pose, rec, pose_valid, Q3, V3, mx, my, lags)
r_prof = ref.latent_lag_profile(X0, pose, rec, pose_valid, Q3, V3, mx, my, lags)
assert prof.shape == (16, 3) and np.allclose(prof, r_prof, atol=1e-6), "lag profile differs from the reference"
best_lag_ms = [lags[i] * 20 for i in prof.argmax(axis=0)]
print("✅ best delay per shared latent (ms):", best_lag_ms, "| peak correlation:", prof.max(axis=0).round(2))

# 📊 What to look for: a clear peak away from 0 ms means the EMG leads; a flat curve means a weak latent.
fig, ax = plt.subplots(figsize=(7, 4))
for i in range(3):
    ax.plot(np.array(lags) * 20, prof[:, i], marker=".", label=f"latent {i + 1}")
ax.set_xlabel("EMG taken this many ms before the pose"); ax.set_ylabel("correlation"); ax.legend()
plt.tight_layout(); plt.show()

✍️ **Q3.** Do all shared latents have the same delay? If a latent peaks at the edge of the
range (300 ms), what would you do next, and what physiological process could produce delays of 150–300 ms (much
longer than the ~50 ms electromechanical delay)?

> *Your answer:*

---
## Task 4 · Is the shared subspace the synergies?
NMF and PCA (notebook 02) keep the EMG directions with the **most variance**. RRR keeps the directions that best
**predict the pose**. Are they the same? Compare subspaces with **principal angles**: for orthonormal bases
`Qa`, `Qb`, the singular values of `Qaᵀ Qb` are the cosines of the angles between the two subspaces (0° = shared
direction, 90° = perpendicular).

Angles between random 3-dimensional subspaces of a 16-dimensional space are *not* 90°; they are often 40–60°.
So build a **null**: angles between the shared subspace and 500 random subspaces. A synergy subspace is "the same"
only if its angles sit **well below** the null.

In [ ]:
def principal_angles(Qa: np.ndarray, Qb: np.ndarray) -> np.ndarray:
    """Principal angles in degrees between the column spaces of two orthonormal bases, smallest first."""
    # TODO
    raise NotImplementedError


def random_subspace(D: int, r: int, rng: np.random.Generator) -> np.ndarray:
    """(D, r) orthonormal basis of a random r-dimensional subspace: QR of a Gaussian matrix."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 4? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. `principal_angles`: singular values of `Qa.T @ Qb` → clip to [−1, 1] (rounding) → arccos → degrees. They come out
   largest-cosine first, i.e. smallest angle first.
2. `random_subspace`: draw a D × r normal matrix and orthonormalise it with QR.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.linalg.svd(M, compute_uv=False)` → only the singular values.
- `np.linalg.qr(A)` → `(Q, R)`; `Q` has orthonormal columns spanning the same space as A.
- `np.degrees(np.arccos(np.clip(s, -1, 1)))`.

```python
Qa = np.eye(3)[:, :1]; Qb = np.eye(3)[:, :1]
np.degrees(np.arccos(np.linalg.svd(Qa.T @ Qb, compute_uv=False)))   # array([0.])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def principal_angles(Qa, Qb):
    s = np.linalg.svd(Qa.T @ ____, compute_uv=False)
    return np.degrees(np.arccos(np.clip(s, ____, ____)))


def random_subspace(D, r, rng):
    Q, _ = np.linalg.qr(rng.normal(size=(____, ____)))
    return Q
```

</details>

In [ ]:
# ✅ Check Task 4
I3 = np.eye(16)[:, :3]
assert np.allclose(principal_angles(I3, I3), 0, atol=1e-5) and np.allclose(principal_angles(I3, np.eye(16)[:, 3:6]), 90)
Qr = random_subspace(16, 3, np.random.default_rng(1))
assert np.allclose(Qr.T @ Qr, np.eye(3), atol=1e-10), "random_subspace must be orthonormal"

_, W3, _ = ref02.nmf_fit(emg[train][::5], 3, n_iter=200, seed=SEED)
Q_nmf = np.linalg.qr(W3.T)[0]                       # synergy subspace (NMF on the envelope)
_, C3 = ref02.pca_fit(X0[train], 3); Q_pca = np.linalg.qr(C3.T)[0]   # top-3 PCs of the log-EMG
ang_nmf, ang_pca = principal_angles(Q3, Q_nmf), principal_angles(Q3, Q_pca)
null = np.array([principal_angles(Q3, random_subspace(16, 3, np.random.default_rng(s))) for s in range(500)])
assert np.allclose(ang_nmf, ref.principal_angles(Q3, Q_nmf), atol=1e-6)
p_nmf = (null[:, 0] <= ang_nmf[0]).mean(); p_pca = (null[:, 0] <= ang_pca[0]).mean()
print(f"✅ angles shared vs NMF {ang_nmf.round(0)}, vs PCA {ang_pca.round(0)}; "
      f"random null first angle: median {np.median(null[:, 0]):.0f}°, 5th percentile {np.percentile(null[:, 0], 5):.0f}°")
print(f"   share of random subspaces at least as close as NMF: {p_nmf:.2f}; as PCA: {p_pca:.2f}")

# 📊 What to look for: are the synergy lines far to the LEFT of the random histogram (= similar), or inside it?
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(null[:, 0], bins=30, color="0.8", label="random subspaces (first angle)")
ax.axvline(ang_nmf[0], color="C2", lw=2, label="NMF synergies"); ax.axvline(ang_pca[0], color="C1", lw=2, label="PCA")
ax.set_xlabel("smallest principal angle to the shared subspace (°)"); ax.legend(); plt.tight_layout(); plt.show()

✍️ **Q4.** Is the shared (pose-relevant) subspace the synergy subspace? What does your answer imply
for notebook 08, where a k = 4 synergy bottleneck did no better than a random one?

> *Your answer:*

---
## Task 5 · "Finger activation" latents: one direction per finger?
For each finger, find the single EMG direction (rank-1 RRR) that best predicts that finger's 4 angles. If each finger
had its own muscles visible at the wrist, the five directions would differ. If the fingers share the long forearm
flexors, they would point the same way. Compare them with `|cos|` (1 = same direction, 0 = perpendicular), and check
against **shifted EMG**, which shows how similar the directions look when the EMG carries no pose information.
That null matters: the four fingers' *poses* move together, so their best EMG directions may look alike for that
reason alone. Only similarity **beyond the null** says something about shared latents.

In [ ]:
FINGERS = {"thumb": [0, 1, 2, 3], "index": [4, 5, 6, 7], "middle": [8, 9, 10, 11],
           "ring": [12, 13, 14, 15], "pinky": [16, 17, 18, 19]}


def finger_directions(X: np.ndarray, Y: np.ndarray, alpha: float = 1.0) -> np.ndarray:
    """(5, D): for each finger (FINGERS order), the unit EMG direction of a rank-1 RRR from X to that finger's
    4 angles, sign-flipped so its largest-magnitude entry is positive."""
    # TODO
    raise NotImplementedError


def abs_cos_matrix(dirs: np.ndarray) -> np.ndarray:
    """(n, n) |cosine| between every pair of rows."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 5? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. For each finger: `rrr_fit(X, Y[:, cols], 1)`, then `predictive_subspace(B_r, 1)[:, 0]` is its direction.
2. Flip the sign so the largest-magnitude entry is positive; stack the five rows.
3. `abs_cos_matrix`: normalise each row to length 1, then the absolute value of the matrix of dot products.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `u * np.sign(u[np.abs(u).argmax()])` flips u if its biggest entry is negative.
- `n = dirs / np.linalg.norm(dirs, axis=1, keepdims=True)`; `n @ n.T` holds every pairwise cosine.

```python
a = np.array([[1.0, 0.0], [1.0, 1.0]])
n = a / np.linalg.norm(a, axis=1, keepdims=True); np.abs(n @ n.T).round(2)   # [[1, 0.71], [0.71, 1]]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def finger_directions(X, Y, alpha=1.0):
    dirs = []
    for cols in FINGERS.values():
        B, _, _, _ = rrr_fit(X, Y[:, ____], 1, alpha)
        u = predictive_subspace(B, 1)[:, 0]
        dirs.append(u * np.sign(u[np.abs(u).argmax()]))
    return np.array(dirs)


def abs_cos_matrix(dirs):
    n = dirs / np.linalg.norm(dirs, axis=1, keepdims=____)
    return np.abs(n @ n.____)
```

</details>

In [ ]:
# ✅ Check Task 5
dirs = finger_directions(X[tr], pose[tr])
assert dirs.shape == (5, 16) and np.allclose(dirs, ref.finger_directions(X[tr], pose[tr]), atol=1e-6)
cosm = abs_cos_matrix(dirs)
assert np.allclose(cosm, ref.abs_cos_matrix(dirs))
cos_null = abs_cos_matrix(finger_directions(X_shift[tr], pose[tr]))
four = cosm[1:, 1:][np.triu_indices(4, 1)]; thumb = cosm[0, 1:]
print(f"✅ |cos| among the four fingers: {four.min():.2f}–{four.max():.2f}; thumb vs fingers: {thumb.min():.2f}–{thumb.max():.2f}")
four_n = cos_null[1:, 1:][np.triu_indices(4, 1)]; thumb_n = cos_null[0, 1:]
print(f"   with shifted EMG (null):  four fingers {four_n.min():.2f}–{four_n.max():.2f}; thumb vs fingers {thumb_n.min():.2f}–{thumb_n.max():.2f}")

# How well does each finger's one direction do on held-out stages? Two scores per finger (rank 1):
#   R²   compares with the held-out stage's OWN average posture, so a wrong average makes it negative;
#   corr is the correlation of predicted and true angles over time: it ignores the average and asks
#        "does the prediction go up and down at the right moments?"
r2_finger, corr_finger = {}, {}
for f, cols in FINGERS.items():
    sc, cc = [], []
    for _, ftr, fte in protocol.leave_one_stage_out(stage_of_rec, rec, pose_valid, filename=filename_of_rec):
        B, mx_, my_, _ = rrr_fit(X[ftr], pose[ftr][:, cols], 1)
        pred_ = rrr_predict(X[fte], B, mx_, my_); true_ = pose[fte][:, cols]
        sc.append(1 - ((true_ - pred_) ** 2).sum() / ((true_ - true_.mean(0)) ** 2).sum())
        cc.append(np.nanmean([np.corrcoef(pred_[:, j], true_[:, j])[0, 1] for j in range(len(cols))]))
    r2_finger[f] = protocol.session_bootstrap_ci(np.array(sc))
    corr_finger[f] = protocol.session_bootstrap_ci(np.array(cc))
for f in FINGERS:
    print(f"   {f:6s} held-out R² {r2_finger[f][0]:+.2f} [{r2_finger[f][1]:+.2f}, {r2_finger[f][2]:+.2f}]   "
          f"corr {corr_finger[f][0]:.2f} [{corr_finger[f][1]:.2f}, {corr_finger[f][2]:.2f}]")

# 📊 What to look for: a bright 4 × 4 block (fingers share a direction) and a darker thumb row.
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a_, M_, title in [(ax[0], cosm, "real EMG"), (ax[1], cos_null, "shifted EMG (null)")]:
    im = a_.imshow(M_, vmin=0, vmax=1, cmap="viridis"); a_.set_title(title)
    a_.set_xticks(range(5), list(FINGERS)); a_.set_yticks(range(5), list(FINGERS))
plt.colorbar(im, ax=ax); plt.show()

✍️ **Q5.** Compare the real |cos| matrix with the **shifted-EMG null**. Does the similarity of the
four fingers' directions tell you they share a "finger activation" latent, or would you see it even with no EMG–pose
link (hint: the four fingers' *poses* are strongly correlated, and the EMG channels are too)? What would be better
evidence for a shared vs a finger-specific latent? Then look at the held-out R² and correlation: why can R² be negative
while the correlation is positive?

> *Your answer:*

---
## Task 6 · Shared vs private: how much of the EMG is about the pose?
The shared subspace is defined by what predicts the pose, not by size. How much of the EMG's variance lives in it?
The rest is **private** to the EMG: co-contraction that doesn't move the hand, wrist and forearm activity, noise.
On the pose side, the columns of `V` show what each shared latent **does** to the hand: a pattern over the 20
joints (an "eigen-movement").

In [ ]:
def variance_in_subspace(X: np.ndarray, Q: np.ndarray) -> float:
    """Share (0–1) of the centred variance of X (T, D) that lies in the column space of orthonormal Q (D, r)."""
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 6? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Centre `X`.
2. Project onto `Q` (`Xc @ Q`): with orthonormal columns, the squared length of the projection is the part of the
   variance inside the subspace.
3. Divide by the total squared length of `Xc`.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- With orthonormal `Q`, `np.sum((Xc @ Q) ** 2)` is the variance captured by the subspace.

```python
Xc = np.array([[1.0, 2.0], [-1.0, -2.0]]); Q = np.array([[1.0], [0.0]])
np.sum((Xc @ Q) ** 2) / np.sum(Xc ** 2)     # 0.2: one fifth of the variance lies along the first axis
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def variance_in_subspace(X, Q):
    Xc = X - X.mean(axis=____)
    return float(np.sum((Xc @ ____) ** 2) / np.sum(Xc ** ____))
```

</details>

In [ ]:
# ✅ Check Task 6
v_shared = variance_in_subspace(X0[train], Q3)
assert abs(v_shared - ref.variance_in_subspace(X0[train], Q3)) < 1e-9
v_pca = variance_in_subspace(X0[train], Q_pca)
v_rand = np.mean([variance_in_subspace(X0[train], random_subspace(16, 3, np.random.default_rng(s))) for s in range(50)])
print(f"✅ share of log-EMG variance in: shared subspace {v_shared:.1%} | top-3 PCs {v_pca:.1%} | random 3-D {v_rand:.1%}")

# 📊 Left: variance shares. Right: what each shared latent does to the hand (columns of V). What to look for: which joints move together in each latent?
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1, 3]})
ax[0].bar(["shared", "top-3 PCs", "random"], [v_shared, v_pca, v_rand], color=["C0", "C1", "0.6"])
ax[0].set_ylabel("share of EMG variance")
for i in range(3):
    ax[1].bar(np.arange(20) + (i - 1) * 0.27, V3[:, i] * np.sign(V3[np.abs(V3[:, i]).argmax(), i]), width=0.27, label=f"latent {i + 1}")
ax[1].set_xticks(range(20), JOINT_NAMES, rotation=90, fontsize=7); ax[1].axhline(0, color="k", lw=0.5)
ax[1].legend(); ax[1].set_title("pose side of each shared latent (columns of V)"); plt.tight_layout(); plt.show()

✍️ **Q6.** What share of the EMG variance is about the pose, and what does the rest represent? Look at
latent 1's pose pattern: which joints does it move, and does it match "close the fingers"?

> *Your answer:*

---
## Task 7 · Pose-informed synergies
Can we keep what makes synergies attractive (non-negative, additive, k numbers) but choose them for the pose?
Add a pose term to the NMF objective:

$$\min_{C,W\ge 0,\;M}\; \lVert X - C W\rVert^2 + \lambda\,\lVert Y_s - C M\rVert^2$$

`X` is the (non-negative) envelope 200 ms before the pose, `Y_s` the centred pose rescaled so the two terms start on a
similar scale, and `M` (k × 20) a linear readout from the activations to the pose. With λ = 0 this is plain NMF.

**The updates**, one round:
- `W ← W ⊙ (CᵀX) / (CᵀC W)`: the usual multiplicative NMF update (the pose term doesn't involve W).
- `M ← (CᵀC)⁻¹ Cᵀ Y_s`: ordinary least squares.
- `C ← max(0, C − ∇/L)` with `∇ = (C W − X) Wᵀ + λ (C M − Y_s) Mᵀ` and `L` the largest eigenvalue of `W Wᵀ + λ M Mᵀ`:
  a **projected gradient** step. The pose term has mixed signs, so a multiplicative update no longer works. Instead,
  take a gradient step of size 1/L (which can't overshoot) and set negative entries to 0.

At test time there is no pose, so the activations come from the EMG alone (`ref02.nmf_transform` with W fixed); the
pose term only shapes **which** synergies W are found. Then a linear map from `log(C)` to the pose is fitted on train.

In [ ]:
def supervised_nmf(X: np.ndarray, Y: np.ndarray, k: int, lam: float = 1.0, n_iter: int = 300, seed: int = 0
                   ) -> tuple[np.ndarray, np.ndarray, np.ndarray, list[float]]:
    """Pose-informed NMF.

    Parameters
    ----------
    X : (T, D) non-negative EMG;  Y : (T, J) pose;  k : synergies;  lam : weight of the pose term

    Start: rng = np.random.default_rng(seed); scale = sqrt(X.mean() / k); C = scale * rng.random((T, k)),
           then W = scale * rng.random((k, D))   (the notebook-02 recipe).
    Ys:    Yc = Y − mean;  Ys = Yc / Yc.std() * sqrt(sum(X²) / sum((Yc / Yc.std())²)).
    Each round, in this order: W update, M update, C update, then append the loss
           sum((X − C W)²) + lam * sum((Ys − C M)²).
    End:   n = norm of each row of W (+1e-10); return C * n, W / n[:, None], M / n[:, None], losses.
    Use 1e-10 in every denominator and as the ridge in the M update.
    """
    # TODO
    raise NotImplementedError

#### 💡 Stuck on Task 7? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. Set up `Ys`, the random start (C first, then W), and an empty loss list.
2. Each round: multiplicative W update; M by solving `(CᵀC + 1e-10 I) M = Cᵀ Ys`; the gradient of the loss with
   respect to C; the step size 1/L; `C = np.maximum(0, C − grad / L)`; append the loss.
3. Normalise the rows of W at the end, scaling C up and M down to compensate.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `np.linalg.eigvalsh(S).max()` → the largest eigenvalue of a symmetric matrix S.
- `np.maximum(0, A)` sets negative entries to 0 (the "projection" onto C ≥ 0).
- Why 1/L? For this loss, a gradient step on C can't increase the loss if the step is at most 1/L.

```python
np.maximum(0, np.array([-1.0, 2.0]))     # array([0., 2.])
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
def supervised_nmf(X, Y, k, lam=1.0, n_iter=300, seed=0):
    rng = np.random.default_rng(seed)
    Yc = Y - Y.mean(axis=0)
    Ys = Yc / Yc.std() * np.sqrt(np.sum(X ** 2) / np.sum((Yc / Yc.std()) ** 2))
    scale = np.sqrt(X.mean() / k)
    C = scale * rng.random((X.shape[0], k))
    W = scale * rng.random((k, X.shape[1]))
    losses, eps = [], 1e-10
    for _ in range(n_iter):
        W *= (C.T @ X) / (C.T @ C @ W + eps)
        M = np.linalg.solve(C.T @ C + eps * np.eye(k), C.T @ ____)
        L = np.linalg.eigvalsh(W @ W.T + lam * M @ M.T).max()
        grad = (C @ W - X) @ W.T + lam * (C @ M - ____) @ M.T
        C = np.maximum(0.0, C - grad / ____)
        losses.append(float(np.sum((X - C @ W) ** 2) + lam * np.sum((Ys - C @ M) ** 2)))
    n = np.linalg.norm(W, axis=1) + eps
    return C * n, W / n[:, None], M / n[:, None], losses
```

</details>

In [ ]:
# ✅ Check Task 7a (a small fit)
Xe = lag_features(emg, rec, [10])                   # (N, 16) envelope 200 ms before the pose (non-negative)
sub = np.where(tr)[0][::20]
C_, W_, M_, L_ = supervised_nmf(Xe[sub], pose[sub], 3, lam=5.0, n_iter=50, seed=SEED)
r_C, r_W, r_M, r_L = ref.supervised_nmf(Xe[sub], pose[sub], 3, lam=5.0, n_iter=50, seed=SEED)
assert (C_ >= 0).all() and (W_ >= 0).all(), "C and W must stay non-negative"
assert len(L_) == 50 and all(b <= a * (1 + 1e-9) for a, b in zip(L_, L_[1:])), "the loss went up: check the step size 1/L"
assert np.allclose(W_, r_W, atol=1e-6), "W differs from the reference: check the order of the updates and the start"
print(f"✅ supervised NMF works: loss {L_[0]:.0f} → {L_[-1]:.0f}")

### Task 7b · Does the pose term help? k = 3, time split
Compare three ways to get k = 3 numbers per frame from the EMG, each followed by a linear map to the pose:
**plain NMF** (λ = 0), **pose-informed NMF** (λ = 5), and **RRR rank 3** (the shared latents, no non-negativity).
Train on every 5th valid training frame (for speed), test on all valid test frames. About 15 s.

In [ ]:
sub = np.where(tr)[0][::5]
k = 3
results_k: dict[str, float] = {}
# TODO: for lam in (0.0, 5.0): W from supervised_nmf(Xe[sub], pose[sub], k, lam, n_iter=300, seed=SEED);
#       activations C_tr = ref02.nmf_transform(Xe[sub], W, n_iter=200), C_te = ref02.nmf_transform(Xe[te], W, n_iter=200);
#       least squares from [log(C + 1e-3), 1] to pose on train; mean absolute error on test →
#       results_k["plain NMF"] (lam 0) and results_k["pose-informed NMF"] (lam 5)
# TODO: results_k["RRR rank 3"]: rrr_fit(X[sub], pose[sub], 3) → error on X[te]
results_k["median pose"] = float(np.abs(pose[te] - np.median(pose[sub], axis=0)).mean())

#### 💡 Stuck on Task 7b? Open one hint at a time

<details>
<summary><b>Hint 1 · the plan in words</b></summary>

1. For each λ: fit, get train and test activations with W fixed, take `log(C + 1e-3)`, add a column of ones, fit
   least squares on train, predict test, record the mean absolute error.
2. RRR: one fit and one prediction with your Task 2 functions.

</details>
<details>
<summary><b>Hint 2 · the tools, with a tiny example</b></summary>

- `A = np.c_[np.log(C + 1e-3), np.ones(len(C))]` adds the intercept column.
- `w = np.linalg.lstsq(A_tr, pose[sub], rcond=None)[0]`, then `A_te @ w`.

```python
np.c_[np.array([[1.0], [2.0]]), np.ones(2)]    # [[1, 1], [2, 1]]
```

</details>
<details>
<summary><b>Hint 3 · a skeleton to complete</b></summary>

```python
for name, lam in [("plain NMF", 0.0), ("pose-informed NMF", 5.0)]:
    _, W, _, _ = supervised_nmf(Xe[sub], pose[sub], k, lam, n_iter=300, seed=SEED)
    C_tr = ref02.nmf_transform(Xe[sub], W, n_iter=200)
    C_te = ref02.nmf_transform(Xe[____], W, n_iter=200)
    A_tr = np.c_[np.log(C_tr + 1e-3), np.ones(len(C_tr))]
    A_te = np.c_[np.log(C_te + 1e-3), np.ones(len(C_te))]
    w = np.linalg.lstsq(A_tr, pose[____], rcond=None)[0]
    results_k[name] = float(np.abs(pose[te] - A_te @ w).mean())

B, mx_, my_, _ = rrr_fit(X[sub], pose[sub], 3)
results_k["RRR rank 3"] = float(np.abs(pose[te] - rrr_predict(X[____], B, mx_, my_)).mean())
```

</details>

In [ ]:
# ✅ Check Task 7b
for key in ["plain NMF", "pose-informed NMF", "RRR rank 3", "median pose"]:
    assert key in results_k, f"missing results_k[{key!r}]"
assert results_k["pose-informed NMF"] <= results_k["plain NMF"] + 0.3, \
    "pose-informed NMF should not be clearly worse than plain NMF: check λ and the readout"
print("✅ held-out error (°), k = 3:", {k_: round(v_, 2) for k_, v_ in results_k.items()})

# 📊 What to look for: how much of the gap between plain NMF and the shared latents does the pose term close?
fig, ax = plt.subplots(figsize=(7, 3.5))
names_ = ["median pose", "plain NMF", "pose-informed NMF", "RRR rank 3"]
ax.barh(names_, [results_k[n_] for n_ in names_], color=["0.6", "C2", "C4", "C0"])
ax.set_xlim(min(results_k.values()) - 1, max(results_k.values()) + 0.5); ax.set_xlabel("test error (°)")
plt.tight_layout(); plt.show()

✍️ **Q7.** Does adding the pose term change which synergies are found, and does that help? Should
notebook 08's bottleneck use plain NMF, pose-informed NMF, or the RRR shared latents, and what do you give up with each
(non-negativity, interpretability, pose information)?

> *Your answer:*

---
## ⚠️ Methodology: what this notebook can and can't tell you
Read this before writing the Result. Each row is a reason the numbers could look better (or different) than the truth, and what would test it.

| Issue | Why it matters | What would test it |
|---|---|---|
| **One person** | Every number is within one participant; shared latents may differ between people | Repeat on several users of the full dataset (stage 7) |
| **15 blocks, 14 stage folds** | CIs are over held-out stages; the two wrists are simultaneous, and the free-style blocks are merged into one fold | Already handled by `protocol` (`filename=`) |
| **Linear maps** | RRR, CCA-like angles and NMF readouts are linear; a curved EMG–pose relation would look higher-rank or weaker | Nonlinear readouts of the same latents (e.g. a small network) in stage 5 |
| **Lag chosen as 200 ms** | The shared subspace is fitted at one lag; Task 3 suggests latents have different delays | Fit with several lags at once (lag_features with [5, 10, 15]) or a full DLAG model |
| **Zero-phase envelope** | The 10 Hz envelope uses about 100 ms of *future* EMG, which shortens the apparent delay | Rebuild the envelope causally before claiming a delay |
| **Pose is camera-tracked** | Tracking noise limits every error; the 6 Hz low-pass removes some jitter but not offsets | Compare per-joint errors with tracking-confidence or repeat tracking |
| **Small gains** | Even the best shared map improves on the median pose by only 1–2°; most pose variance is not predicted | Report R² per finger and per stage, not only mean degrees |

---
## Result · gate for stage 4b
**Question:** Stage 4b: how many dimensions do the EMG and the pose share, and are they the synergies?

| Measure | Value (mean, 95% CI over sessions) | Baseline / null | Pass? |
|---|---|---|---|
| Chosen rank (Task 2b) and its error − median pose | | | |
| Rank 3 − shifted EMG (paired CI) | | | |
| Best delay per shared latent (Task 3) | | | |
| Smallest principal angle to NMF / PCA vs random null (Task 4) | | | |
| |cos| among the four fingers; thumb vs fingers (Task 5) | | | |
| Share of EMG variance in the shared subspace (Task 6) | | | |
| Pose-informed NMF vs plain NMF vs RRR, k = 3 (Task 7) | | | |

**Gate:** the shared subspace beats the shifted-EMG null on held-out stages with rank ≤ 3, and its angles to the synergies fall inside (or near) the random null, i.e. it is a different subspace from the variance synergies.

- **Pass →** stage 5 (notebook 07): model the dynamics of the shared latents; stage 6 (notebook 08): build the bottleneck from the shared latents or pose-informed synergies.
- **Fail →** if rank ≤ 3 doesn't beat the null, the EMG–pose link is too weak for a latent model in one person: report it and go straight to stage 7 thinking with this caveat; if the shared subspace *is* the synergies, keep plain NMF in stage 6.
- **Was your prediction right? Why or why not?**